In [ ]:
# ============================================================
# SMART AGRICULTURE — 5 ML PROBLEMS
# Clean End-to-End ML Pipeline
# ============================================================

# If using Jupyter only:
# %matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import pickle
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, IsolationForest
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, classification_report, confusion_matrix

sns.set_theme(style="whitegrid")

# ============================================================
# FUNCTIONS
# ============================================================

def regression_models(models, X_train, X_test, y_train, y_test, X, y):
    results = {}

    for name, model in models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        mae = mean_absolute_error(y_test, pred)
        r2 = r2_score(y_test, pred)

        cv = cross_val_score(model, X, y, cv=5)

        results[name] = mae

        print(f"{name:20s} MAE={mae:.2f}  R2={r2:.4f}  CV={cv.mean():.4f}")

    best = min(results, key=results.get)

    print("Best Model:", best)

    return best


def feature_importance(model, features, title):
    importance = model.feature_importances_

    df_imp = pd.DataFrame({
        "Feature": features,
        "Importance": importance
    }).sort_values("Importance", ascending=False)

    plt.figure(figsize=(8, 5))
    sns.barplot(x="Importance", y="Feature", data=df_imp)
    plt.title(title)
    plt.show()


def shap_explain(model, X_test):
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)

    shap.summary_plot(shap_values, X_test)
    shap.summary_plot(shap_values, X_test, plot_type="bar")


def classifier_results(model, X_test, y_test):
    pred = model.predict(X_test)

    acc = accuracy_score(y_test, pred)

    print("Accuracy:", acc)
    print(classification_report(y_test, pred))

    cm = confusion_matrix(y_test, pred)

    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, cmap="Blues", fmt="d")
    plt.title("Confusion Matrix")
    plt.show()

    return acc


def save_model(model, filename):
    with open(filename, "wb") as f:
        pickle.dump(model, f)

    print("Saved:", filename)


# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv("paddydataset.csv")
df.columns = df.columns.str.strip()
df = df.dropna()

print("Dataset shape:", df.shape)

# ============================================================
# ENCODE CATEGORICAL DATA
# ============================================================

df_enc = df.copy()

# Better: separate encoder for each column
label_encoders = {}

for col in df_enc.select_dtypes(include="object").columns:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))
    label_encoders[col] = le

# ============================================================
# PROBLEM 1 — YIELD PREDICTION
# ============================================================

print("\nPROBLEM 1 — Yield Prediction")

target_col = "Paddy yield(in Kg)"

if target_col not in df_enc.columns:
    raise ValueError(f"Column '{target_col}' not found in dataset.")

X = df_enc.drop(target_col, axis=1)
y = df_enc[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(),
    "RF": RandomForestRegressor(n_estimators=100, random_state=42)
}

best_model_name = regression_models(models, X_train, X_test, y_train, y_test, X, y)

# Hyperparameter tuning
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None]
}

grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)

model_p1 = grid.best_estimator_

pred1 = model_p1.predict(X_test)

mae1 = mean_absolute_error(y_test, pred1)
r21 = r2_score(y_test, pred1)

print("MAE:", mae1)
print("R2 :", r21)

feature_importance(model_p1, X.columns, "Yield Feature Importance")

# Optional: SHAP can be slow / version-sensitive
try:
    shap_explain(model_p1, X_test)
except Exception as e:
    print("SHAP skipped due to error:", e)

save_model(model_p1, "model_p1_yield.pkl")

# ============================================================
# PROBLEM 2 — VARIETY RECOMMENDATION
# ============================================================

print("\nPROBLEM 2 — Variety Recommendation")

if "Variety" not in df.columns:
    print("Skipping Problem 2: 'Variety' column not found.")
    acc2 = 0
    model_p2 = None
else:
    var_enc = LabelEncoder()

    df2 = df_enc.copy()
    df2["var_target"] = var_enc.fit_transform(df["Variety"].astype(str))

    X2 = df2.drop(["Variety", "var_target", target_col], axis=1, errors="ignore")
    y2 = df2["var_target"]

    X_train, X_test, y_train, y_test = train_test_split(
        X2, y2, test_size=0.2, stratify=y2, random_state=42
    )

    model_p2 = RandomForestClassifier(n_estimators=100, random_state=42)

    model_p2.fit(X_train, y_train)

    acc2 = classifier_results(model_p2, X_test, y_test)

    save_model(model_p2, "model_p2_variety.pkl")

# ============================================================
# PROBLEM 3 — FERTILIZER OPTIMIZATION
# ============================================================

print("\nPROBLEM 3 — Fertilizer Optimization")

fert_cols = [
    "DAP_20days",
    "Urea_40Days",
    "Potassh_50Days",
    "Micronutrients_70Days",
    "Weed28D_thiobencarb",
    "Pest_60Day(in ml)"
]

features = ["Hectares", "Agriblock", "Variety", "Soil Types"]
features = [c for c in features if c in df_enc.columns]

fert_models = {}
mae_fert = []

available_fert_cols = [c for c in fert_cols if c in df_enc.columns]

if not features or not available_fert_cols:
    print("Skipping Problem 3: Required fertilizer columns or features missing.")
else:
    for col in available_fert_cols:
        X3 = df_enc[features]
        y3 = df_enc[col]

        X_train, X_test, y_train, y_test = train_test_split(
            X3, y3, test_size=0.2, random_state=42
        )

        model = RandomForestRegressor(n_estimators=100, random_state=42)

        model.fit(X_train, y_train)

        pred = model.predict(X_test)

        mae = mean_absolute_error(y_test, pred)
        mae_fert.append(mae)

        print(col, "MAE:", mae)

        fert_models[col] = model

    save_model(fert_models, "model_p3_fertilizer.pkl")

# ============================================================
# PROBLEM 4 — PREHARVEST YIELD
# ============================================================

print("\nPROBLEM 4 — Preharvest Yield Prediction")

required_cols_p4 = ["Trash(in bundles)", "Hectares", target_col]

if all(col in df.columns for col in required_cols_p4):
    X4 = df[["Trash(in bundles)", "Hectares"]]
    y4 = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X4, y4, test_size=0.2, random_state=42
    )

    model_p4 = RandomForestRegressor(n_estimators=100, random_state=42)

    model_p4.fit(X_train, y_train)

    pred4 = model_p4.predict(X_test)

    mae4 = mean_absolute_error(y_test, pred4)
    r24 = r2_score(y_test, pred4)

    print("MAE:", mae4)
    print("R2 :", r24)

    save_model(model_p4, "model_p4_preharvest.pkl")
else:
    print("Skipping Problem 4: Required columns missing.")
    mae4 = 0

# ============================================================
# PROBLEM 5 — ANOMALY DETECTION
# ============================================================

print("\nPROBLEM 5 — Farm Anomaly Detection")

df5 = df_enc.copy()

scaler = StandardScaler()

X5 = scaler.fit_transform(df5)

model_p5 = IsolationForest(contamination=0.1, random_state=42)

df5["anomaly"] = model_p5.fit_predict(X5)

anomaly_count = (df5["anomaly"] == -1).sum()

print("Anomalies detected:", anomaly_count)

save_model(model_p5, "model_p5_anomaly.pkl")
save_model(scaler, "scaler_p5_anomaly.pkl")

# ============================================================
# FINAL MODEL COMPARISON TABLE
# ============================================================

fert_score = round(np.mean(mae_fert), 2) if len(mae_fert) > 0 else 0

results = pd.DataFrame({
    "Problem": [
        "Yield Prediction",
        "Variety Recommendation",
        "Fertilizer Optimization",
        "Preharvest Yield",
        "Anomaly Detection"
    ],
    "Best Model": [
        "Random Forest",
        "Random Forest",
        "Random Forest",
        "Random Forest",
        "Isolation Forest"
    ],
    "Metric": [
        "MAE",
        "Accuracy",
        "MAE",
        "MAE",
        "Anomaly Count"
    ],
    "Score": [
        round(mae1, 2),
        round(acc2, 3),
        fert_score,
        round(mae4, 2),
        anomaly_count
    ]
})

print("\nFINAL MODEL COMPARISON")
print(results)

# ============================================================
# PERFORMANCE CHART
# ============================================================

plt.figure(figsize=(8, 5))
sns.barplot(x="Problem", y="Score", data=results)
plt.xticks(rotation=30)
plt.title("Final Model Performance Comparison")
plt.ylabel("Score")
plt.xlabel("Problems")
plt.show()

d:\python\python v\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FileNotFoundError: [Errno 2] No such file or directory: 'paddydataset.csv'